# Student Performance Analysis and Prediction

This notebook reuses the project modules. Passing is defined as G3 >= 10. The UCI mathematics CSV is observational data; results do not establish causation.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('Run this notebook from the project or notebooks directory.')
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_PATH, FIGURES_DIR, MODEL_DIR
from src.download_data import download_data
from src.eda import run_eda
from src.preprocessing import load_and_prepare
from src.train_model import train_model
from src.evaluate_model import evaluate_model
from src.generate_report import generate_report

print('Project imports ready:', PROJECT_ROOT)

## Load data

In [ ]:
download_data()
data = pd.read_csv(DATA_PATH, sep=';', encoding='utf-8')
data.shape

## Understanding the data
Inspect rows, types, descriptive statistics, missing values, duplicates, grade validity, and the chosen class counts.

In [ ]:
display(data.head())
display(data.tail())
display(data.dtypes.to_frame('dtype'))
display(data.describe(include='all').T)
display(data.isna().sum().to_frame('missing'))
print('duplicates:', data.duplicated().sum())
print('invalid grades:', (~data.G3.between(0, 20)).sum())
display(data.G3.ge(10).map({True: 'pass', False: 'fail'}).value_counts())

## Preprocessing and split
Exact duplicates are removed. Missing rows are dropped only when present. G1/G2/G3 are excluded to prevent direct same-course grade leakage. The transformer is fitted to training rows only.

In [ ]:
prepared = load_and_prepare()
print(prepared.X_train.shape, prepared.X_val.shape, prepared.X_test.shape)
print('Predictor columns:', prepared.feature_names)

## EDA and the five required figures
The EDA module also saves class distribution, numeric summaries, categorical counts, correlations, IQR outlier counts, a Kruskal-Wallis test, and interpretations computed from data.

In [ ]:
eda_results = run_eda(data)
for filename in ['g3_histogram_kde.png', 'g3_by_studytime_boxplot.png', 'numeric_correlation_heatmap.png', 'absences_vs_g3.png', 'mean_g3_by_studytime.png']:
    display(Image(filename=str(FIGURES_DIR / filename)))

## Seven visualization stages
1. **Acquire:** download the public UCI data and verify its shape.
2. **Parse:** read the semicolon-delimited CSV and inspect schema.
3. **Filter:** examine duplicates, missingness, grade bounds, and IQR candidates; do not delete legitimate outliers.
4. **Mine:** compute summaries, correlations, class counts, and a non-causal group test.
5. **Represent:** create the five EDA figures and class chart.
6. **Refine:** use readable titles/axes and export PNG at 200 dpi.
7. **Interact:** save a Plotly chart with hover data and a dropdown to change grouping.

In [ ]:
display(Image(filename=str(FIGURES_DIR / 'pass_fail_distribution.png')))
print('Interactive Plotly chart:', FIGURES_DIR / 'interactive_g3.html')
print('Kruskal-Wallis p-value:', eda_results['kruskal_p_value'])

## Model and hyperparameter comparison
A Keras MLP uses Dense(64, relu), Dropout(0.3), Dense(32, relu), Dropout(0.2), and Dense(1, sigmoid). The three configurations are selected by validation loss only; the test set is not consulted.

In [ ]:
training = train_model(prepared)
training['selected_setting'], training['best_validation_loss']

## Validation permutation feature importance
The project evaluation function computes importance by permuting original validation columns. The result is predictive association, not causal importance.

In [ ]:
metrics = evaluate_model()
display(Image(filename=str(MODEL_DIR / 'permutation_importance.png')))
metrics

## Evaluation and conclusion
The test subset is only about 60 students, so its metrics are noisy. Correlation is not causation, the dataset represents one school, and predictions must not drive real educational decisions without validation, fairness checks, and human oversight. Future work includes cross-validation, other models, and more representative data.

In [ ]:
display(pd.Series({key: metrics[key] for key in ['test_samples', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']}))
display(Image(filename=str(MODEL_DIR / 'loss_curves.png')))
display(Image(filename=str(MODEL_DIR / 'accuracy_curves.png')))
display(Image(filename=str(MODEL_DIR / 'confusion_matrix.png')))
display(Image(filename=str(MODEL_DIR / 'roc_curve.png')))
generate_report()